In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("w14Alab.ipynb")

---

<h3><center>E7 -  Introduction to Programming for Scientists and Engineers</center></h3>

<h2><center>Lab session #14-A <br></center></h2>

<h1><center>Simulating simple heating and cooling<br></center></h1>

---

In [ ]:
from resources.hashutils import *
import numpy as np
import matplotlib.pyplot as plt

# Part 1: Exponential cooling

Newton's law of cooling states that "The rate of heat loss of a body is directly proportional to the difference in the temperatures between the body and its environment." [[Wikipedia](https://en.wikipedia.org/wiki/Newton%27s_law_of_cooling)]. Assuming that the "rate of heat loss" is proportional to the rate of temperature loss, we immediately obtain a first order ordinary differential equation (ODE) for the temperature of the body:

\begin{equation}
T'(t) = -\alpha(T(t)-T_a)
\end{equation} 

where, 

+ $T$ is the temperature of the body, measured in degrees centrigrade,
+ $t$ is time, measured in seconds,
+ $T_a$ is the ambient temperature, measured in degrees centigrade, and
+ $\alpha$ is a positive coefficient of proportionality, related to various physical quantities: the heat capacitance of the body, the heat transfer coefficient between the body and the environment, and the total exposed area of the body. $\alpha$ has units of 1/seconds.

This ODE is similar to one that was shown in lecture: $y'+y=0$, whose solutions are of the form $y(t)=Ce^{-t}$. Fixing the initial condition to be $y(0)=y_0$, we found a unique solution to the initial value problem: $y(t)=y_0 \:e^{-t}$ for all $t\geq 0$.

We can similarly find a solution to the cooling law with initial condition $T(0)=T_0$:

\begin{equation*}
T(t) = T_a + (T_0-T_a)e^{-\alpha t}\quad \forall\; t\geq 0
\end{equation*}

This solution is given here without proof, since techniques for deriving it are outside of the scope of this class. A plot of $T(t)$ with $\alpha=0.001$ and $T_a=22$ is shown below.

<img src="resources/expdecay.png" width=600>

Having found an exact solution to the initial value problem, we would typically not bother to construct a numerical solution. We will nevertheless do so here in order to test our numerical methods on a simple ODE.

By the way, the name "cooling law" can be a bit confusing, since the formula also applies when the ambient temperature $T_a$ is *higher* than $T(t)$, and hence the body is heating up. 


## Part 1.1: State equation

To construct our numerical solution, we will need a function that returns the right-hand side of the cooling law (Eq. (1)), given the values of $\alpha$, $T_a$, and $T$. Write a function called `cooling_law(T,Ta)` that takes a temperature `T`, and the ambient temperature `Ta`. `alpha` will be treated as a global constant, and therefore does not need to be included in the argument list. .

The function should return the right hand side of Eq. (1). 

**Note**
+ We will use `alpha=1/7000` throughout the lab. This is a realistic value for one liter of water. 

In [ ]:
alpha = 1/7000
...

In [ ]:
grader.check("p1p1")

## Part 1.2: Time grid

Create a function called `get_time_grid(ts,h)` that takes the total simulation time `ts` (in seconds) and the time step `h` (in seconds), and returns `tvec` and `K` where
+ `tvec` is a NumPy array of time instants ranging from 0 to `ts` (excluded), with a step size of `h`.
+ `K` is the number of elements in `tvec`.

In [ ]:
...

In [ ]:
# Test your code
...

In [ ]:
grader.check("p1p2")

## Part 1.3: Exact solution

Write a function called `evaluate_sol(t,T0,Ta)` that takes
+ `t`: a scalar or NumPy array of time values,
+ `T0`:  an initial temperature (in degrees centigrade),
+ `Ta`: the ambient temperature,
and returns the exact solution of the cooling IVP given in the introduction. The return value should bE a 1D NumPy array of the same length as `t`. 

**Note** 
+ You may assume that `alpha` is in the global scope. 

In [ ]:
...

In [ ]:
grader.check("p1p3")

## Part 1.4: Euler's method

Write a function called `euler_solver(ts,h,T0,Ta)` that takes inputs `ts`, `h`, `T0`, and `Ta` described in previous parts, and returns the tuple `tvec, That`, where
+ `tvec` is an array of time instants, as returned by `get_time_grid`, and
+ `That` is the numerical solution computed with Euler's method to the initial value problem for Eq. (1).

**Note**:
+ `That` should be a 1D NumPy array with length `K`. The first entry in `That` should be `T0`. 
+ `That[k]` corresponds to $\hat{y}[k]$ in the lecture slides. 
+ You are encouraged to try to write the function from scratch. However you may also use the template code provided in `resources/euler_template.ipynb`.

In [ ]:
...

In [ ]:
grader.check("p1p4")

## Example usage and plot

The next cell computes and plots the exact and approximate solutions (`Tsol` and `That`) with initial temperature of 100&deg;C, a 5-hour simulation time, and a ten-minute time step. 

In [ ]:
ts=5*3600  # 5 hours
h=10*60    # 10 minutes
T0=100
Ta=22

tvec, That = euler_solver(ts,h,T0,Ta)
Tsol = evaluate_sol(tvec,T0,Ta)

tvec_hr = tvec/3600

_, ax = plt.subplots(nrows=2,figsize=(6,4),sharex=True)
ax[0].plot(tvec_hr,Tsol,color='k',marker='.',markersize=2,linewidth=1,label=r'$T[k]$')
ax[0].plot(tvec_hr,That,color='m',marker='.',markersize=2,linewidth=1,label=r'$\hat{T}[k]$')
ax[0].axhline(Ta,color='g',linewidth=2,linestyle=':',label=r'$T_a$')
ax[0].legend(fontsize=12)
ax[0].grid()
ax[0].spines[['top','right']].set_visible(False)
ax[0].set_ylabel(r'$^\circ$C',fontsize=12)

ax[1].plot(tvec_hr,np.abs(Tsol-That),color='b',markersize=2,linewidth=2,label=r'$|T[k]-\hat{T}[k]|$')
ax[1].legend(fontsize=12)
ax[1].grid()
ax[1].set_xlabel('time [hr]',fontsize=12)
ax[1].set_ylabel(r'$^\circ$C',fontsize=12)
ax[1].spines[['top','right']].set_visible(False)

## Part 1.5: Dependence of the error on the time step

Use the functions that you've written so far to evaluate the error of Euler's method for a range of step sizes.

Your code should produce these variables:
+ `hs` ... a 1D array of time step size values ranging from 10 seconds (included), to 300 seconds (excluded), in increments of 10 seconds. 
+ `errors` ... a 1D array holding the error values corresponding to each time step in `hs`.

We will define the error for a particular step size $h$ as the largest absolute deviation $|\hat{T}[k]-T[k]|$ recorded in the simulation using that value of $h$. That is,
\begin{equation*}
e = \max_{k:0... K-1} \Big| \hat{T}[k]-T[k] \Big| 
\end{equation*}
This, for example, would be highest point reached by the blue line in the figure above.

For each time step `h`, run `euler_solver` with 
+ an initial temperature of 100 &deg;C. 
+ an ambient temperature of 22 &deg;C. 
+ a total simulation time of 5 hours.

In [ ]:
...

In [ ]:
_, ax = plt.subplots(figsize=(6,3))
ax.plot(hs,errors,color='darkred',linewidth=1,marker='.')
ax.set_ylabel('max error [C]',fontsize=12)
ax.set_xlabel('time step h [sec]',fontsize=12)
ax.spines[['top','right']].set_visible(False)
ax.spines[['bottom','left']].set_position('zero')
ax.grid()

In [ ]:
grader.check("p1p5")

# Part 2: Generic IVP solver

As discussed in lecture, a general initial value problem (IVP) is a problem of the form,
\begin{align*}
&\mathbf{y}'(t)= f(\mathbf{y},t) \\
&\mathbf{y}(0)=  \mathbf{y}_0
\end{align*}
where $\mathbf{y}$ is a vector of variables, $f$ is the *state equation*, and $t$ is time. Many numerical methods used for approximating solutions to IVPs share a common structure: they advance  through time adding increments of $h\:\mathbf{\hat{s}}$, 
\begin{equation*}
\mathbf{\hat{y}}[k+1] = \mathbf{\hat{y}}[k] + h\: \mathbf{\hat{s}}
\end{equation*}
where $\mathbf{\hat{s}}$ is an estimate of the slope.
From this perspective, Euler's method, the midpoint method, and Heun's method, can be understood simply as different approaches to computing $\mathbf{\hat{s}}$. Here we will use this observation to build a concise implementation of these three IVP solvers. 

**Note**: To keep things simple, we will limit our implementation to first order ODEs. In other words, we will assume that $\mathbf{y}(t)$ is a *scalar*, not a *vector*. Thus we have that $\mathbf{\hat{y}}[k]$ and $\mathbf{\hat{s}}$ are scalars as well.

## Part 2.1: Implement a generic initial value problem solver
Write a function called `ivp_solver(f,ts,h,y0,shat_estimator)`, that takes,

+ `f` : A *Python function* that evaluates the state equation. (The input/output specification of this function will not matter until Part 2.2)
+ `ts`: The total simulation time in seconds.
+ `h` : The time step in seconds.
+ `y0` : The initial condition. Again, we will assume that this is a scalar and not an array.  
+ `shat_estimator(f,h,yhatk,t)`: A *Python function* that evaluates the slope of the numerical solution, $\hat{s}$. This function takes arguments `(f,h,yhatk,t)`, where `f` and `h` are as defined above, `yhatk` is the estimate at time step $k$, and `t` is the current time in seconds. 

This function should implement the generic IVP solver ($\mathbf{\hat{y}}[k+1] = \mathbf{\hat{y}}[k] + h\: \mathbf{\hat{s}}$), and it should return `tvec` and `yhat`, in that order.

**Notes**:
+ You are encouraged to try to write the function from scratch. However you may also use the template provided in `resources/ivp_template.ipynb`.
+ Notice that `shat_estimator` takes as input the continuous time $t$. This can be obtained as $t=kh$.

In [ ]:
...

In [ ]:
grader.check("p2p1")

## Part 2.2: Euler's method

We will implement the three numerical methods -- Euler's method, the midpoint method, and Heun's method -- as three different versions of `shat_estimator(f,h,yhatk,t)`. We begin with Euler's method, which is the simplest one. 

Create a function called `shat_euler` that takes the same arguments as `shat_estimator` and returns the value of $\mathbf{\hat{s}}$ according to Euler's method. That is, it computes $\mathbf{\hat{s}}$ with

\begin{equation*}
\mathbf{\hat{s}} = f(\mathbf{\hat{y}}[k],t)
\end{equation*}

Notice that `h` is not used in Euler's method. It must nevertheless be included as an input argument, since the argument list for `shat_euler` must match that of `shat_estimator`.

In [ ]:
...

In [ ]:
grader.check("p2p2")

## Part 2.3: Midpoint method

Create a function called `shat_midpoint(f,h,yhatk,t)`, analogous to `shat_euler(f,h,yhatk,t)`, but for the midpoint method. See the lecture recording for details. 

In [ ]:
...

In [ ]:
grader.check("p2p3")

## Part 2.4: Heun's method

Create a function called `shat_heun(f,h,yhatk,t)` that computes the slope according to the Heun's method. See the lecture recording for details. 

In [ ]:
...

In [ ]:
grader.check("p2p4")

## Part 2.5: Time-varying ambient temperature

Next we will apply our three numerical methods to the cooling law from Part 1. To make it more interesting, we will allow the ambient temperature $T_a$ to vary in time. 

Let's suppose that the body whose temperature is being predicted is a pot of water that is repeatedly placed on a flame and removed. Suppose the applied profile of ambient temperature $T_a(t)$ is as shown below. 

<img src="resources/Ta.png" width=600>

This profile can be expressed mathematically as follows:

\begin{equation}
T_a(t) =
    \begin{cases}
      200 & \text{if } t\%1800<1200 \\
      22 & \text{otherwise}
    \end{cases}       
\end{equation}

Create a function called `get_Ta(t)` that computes this function for a given *scalar* value `t` (not a NumPy array).

In [ ]:
...

In [ ]:
grader.check("p2p5")

## Part 2.6: Implement a time-varying version of `cooling_law`

We will also need an updated version of the `cooling_law` function that includes time `t` an input argument. The time will be used to get the ambient temperature `Ta` using `get_Ta(t)`, and hence we will no longer need `Ta` as an input argument for the cooling law. 

Create a function called `cooling_law_with_time(T,t)` that takes a value of temperature `T` in degrees centrigrade and a value of time `t` in seconds. The function should return the right-hand side of the time-dependent state equation:
\begin{equation*}
f(T,t) =  -\alpha(T-T_a(t))
\end{equation*}

In [ ]:
def cooling_law_with_time(T,t):
    ...

In [ ]:
grader.check("p2p6")

## Part 2.7: Putting it all together

We now have all of the necessary components to compute a numerical solution of the initial value problem using three numerical methods: Euler's method, the midpoint method, or Heun's method. 

Use `ivp_solver` to "simulate" the temperature of a pot of water as it is moved on and off a flame following the profile $T_a(t)$ of Part 2.5. 

The simulation should have these characteristic:
+ 5-hour simulation time.
+ 5-minute time step.
+ Initial condition of $0^\circ C$.

Compute the solution using each of the three numerical methods, and save the results as `T_euler`,`T_midpoint`, and `T_heun`.

**Note**

+ Although it is not evaluated, you should try running it with smaller values of `h` and noticing how all three methods approach a single exact solution. 

In [ ]:
...
tvec, T_euler    = ...
_, T_midpoint = ...
_, T_heun     = ...

In [ ]:
Ta = [get_Ta(t) for t in tvec]

_, ax = plt.subplots(figsize=(8,3))
ax.plot(tvec,Ta,linestyle='None',marker='.',markersize=4,color='purple',label='$T_a$')
ax.plot(tvec,T_euler,linewidth=1,label='Euler')
ax.plot(tvec,T_midpoint,linewidth=1,label='Midpoint')
ax.plot(tvec,T_heun,linewidth=1,marker='.',markersize=1,label='Heun')
ax.tick_params('both',labelsize=10)
ax.spines[:].set_visible(False)
ax.legend(loc='upper right',fontsize=10)
ax.grid()

In [ ]:
grader.check("p2p7")

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

Make sure you submit the .zip file to Gradescope.

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)